# Phase 5 · Validate pair-aware release tables

A release joins normalized pathogen, host, article, and evidence tables. Supported V4-native pairs must have PMID links, nonempty evidence spans, and the disease relationship context. Legacy rows retain their provenance and are counted separately.

This notebook validates the imported staging evidence as a dry-run example; it does not promote the tables or create a native V4 release.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('06_release_validation', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 11:29:55,107 INFO Notebook session started at 2026-11-02T11:29:55.107923+00:00


2026-11-02 11:29:55,108 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 11:29:55,108 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/06_release_validation.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/06_release_validation.log


## Step 1 · Load candidate staging tables

In [2]:
import pandas as pd
from IPython.display import display
from pathogen_atlas_v4.release import validate_pair_release, evidence_metrics
STAGING_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/'))[-1]
pathogens = pd.read_parquet(STAGING_ROOT / 'taxonomy/pathogens.parquet')
hosts = pd.read_parquet(STAGING_ROOT / 'taxonomy/hosts.parquet')
articles = pd.read_parquet(STAGING_ROOT / 'literature/articles.parquet')
evidence = pd.read_parquet(STAGING_ROOT / 'evidence/pathogen_host_evidence.parquet')
print('Staging:', STAGING_ROOT)
print('Pathogens:', f'{len(pathogens):,}', '| hosts:', len(hosts), '| articles:', f'{len(articles):,}', '| pair evidence:', f'{len(evidence):,}')
LOG.info('Release validation inputs loaded')

2026-11-02 11:29:56,153 INFO Release validation inputs loaded


Staging: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/staging/v3-to-v4-20261102-ncbi-f278697a8ba6-v3
Pathogens: 142,201 | hosts: 5 | articles: 1,733,750 | pair evidence: 2,674


## Step 2 · Run foreign-key and positive-span validation

In [3]:
validate_pair_release(pathogens, hosts, articles, evidence)
print('Pair-release integrity checks passed.')
LOG.info('Pair release foreign keys and positive-span checks passed')

2026-11-02 11:29:56,333 INFO Pair release foreign keys and positive-span checks passed


Pair-release integrity checks passed.


## Step 3 · Summarize provenance and pair outcomes

In [4]:
display(evidence.groupby(['provenance_status','decision'], dropna=False).size().rename('records').to_frame())
print('Unique supported pathogen–host pairs:', evidence.loc[evidence.decision.eq('supported'), ['pathogen_id','host_id']].drop_duplicates().shape[0])
LOG.info('Evidence outcomes: %s', evidence.groupby(['provenance_status','decision']).size().to_dict())

,,records
provenance_status,decision,
legacy_v3_accepted,supported,2674


2026-11-02 11:29:56,341 INFO Evidence outcomes: {('legacy_v3_accepted', 'supported'): 2674}


Unique supported pathogen–host pairs: 440


## Step 4 · Build pair-level literature metrics

In [5]:
matches = pd.read_parquet(STAGING_ROOT / 'literature/pathogen_article_matches.parquet')
# The migration table is pathogen-scoped; host-aware pair metrics are only computed for host-paired records.
legacy_pairs = evidence[['pathogen_id','host_id','pmid','decision']].drop_duplicates()
legacy_pair_counts = legacy_pairs.groupby(['host_id','decision']).size().rename('pair_evidence_rows').to_frame()
display(legacy_pair_counts)
print('Imported pathogen-only matches:', f'{len(matches):,}')
print('Native pair search metrics are produced after the host-aware PubMed phase.')
LOG.info('Pair evidence summary generated; migration matches=%d', len(matches))

,,pair_evidence_rows
host_id,decision,
ncbi:9606,supported,2674


2026-11-02 11:29:57,058 INFO Pair evidence summary generated; migration matches=3450866


Imported pathogen-only matches: 3,450,866
Native pair search metrics are produced after the host-aware PubMed phase.


## Step 5 · Release decision

In [6]:
BUILD_NATIVE_RELEASE = False
if BUILD_NATIVE_RELEASE:
    raise RuntimeError('Build a native release only after pair-aware acquisition, disease-gate review, and profile completeness checks.')
print('Validation only. No tables were promoted or labelled as a native V4 release.')
LOG.info('Native release build skipped')

2026-11-02 11:29:57,061 INFO Native release build skipped


Validation only. No tables were promoted or labelled as a native V4 release.
